In [ ]:
import sqlite3
import pandas as pd

In [ ]:
DB_PATH = 'data/sharadar.db'

In [ ]:
with sqlite3.connect(DB_PATH) as conn:
    tables = conn.execute("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name").fetchall()
tables

In [ ]:
ticker = 'AAPL'
pd.read_sql_query('SELECT * FROM tickers WHERE ticker = ?', sqlite3.connect(DB_PATH), params=(ticker,))

In [ ]:
pd.read_sql_query('SELECT * FROM actions WHERE ticker = ?', sqlite3.connect(DB_PATH), params=(ticker,)).head()

In [ ]:
pd.read_sql_query('SELECT * FROM sp500 WHERE ticker = ?', sqlite3.connect(DB_PATH), params=(ticker,)).head()

In [ ]:
pd.read_sql_query('SELECT * FROM fundamentals WHERE ticker = ?', sqlite3.connect(DB_PATH), params=(ticker,)).head()

# Net Share Issuance
Standalone V2 research notebook.

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
START = '2000-01-01'; END = '2024-12-31'
sys.path.insert(0, str(Path.cwd()))
from strategies.net_share_issuance import run

Net share issuance measures the change in basic shares outstanding. Low issuance (repurchases) is expected to outperform high issuance, using only filings known by each rebalance date.

In [ ]:
result = run(DB_PATH, START, END)

In [ ]:
print('NET SHARE ISSUANCE — FULL UNIVERSE')
print('Period:', result.n_months, 'months')
print('Alpha:', result.alpha_annual, 't/p:', result.alpha_tstat, result.alpha_pvalue)
print('Betas:', result.betas)
print(((1 + result.quintile_returns).prod() ** (12 / len(result.quintile_returns)) - 1).to_string())

In [ ]:
annual = (1 + result.quintile_returns).prod() ** (12 / len(result.quintile_returns)) - 1
annual.plot(kind='bar', color=['green','lightgray','lightgray','lightgray','red'], title='Net Share Issuance — Quintile Returns')
plt.show()

In [ ]:
q1 = (1 + result.monthly_returns).cumprod()
ew = result.portfolio_history.groupby('date')['return'].mean().add(1).cumprod()
pd.concat({'Q1': q1, 'Equal-weight universe': ew}, axis=1).plot(title='Cumulative returns')
plt.show()

In [ ]:
pd.Series(result.betas).plot(kind='barh', title='Factor decomposition')
plt.show()

In [ ]:
pd.DataFrame(result.size_terciles or {}).T

## Interpretation
What does this mean? Does it match the hypothesis?

## Quality
Quality combines profitability, operating efficiency, leverage, interest coverage, and earnings quality. We expect the signal to load most strongly on RMW and possibly be defensive, especially among smaller firms.

In [ ]:
from strategies.quality import run as run_quality
result_quality = run_quality(DB_PATH, START, END)

In [ ]:
print('QUALITY ? FULL UNIVERSE')
print('Period:', result_quality.n_months, 'months')
print('Dimensions:', result_quality.dimensions_used)
print('Alpha:', result_quality.alpha_annual, 't/p:', result_quality.alpha_tstat, result_quality.alpha_pvalue)
print('Betas:', result_quality.betas)
print('Lookahead violations:', result_quality.lookahead_violations)

In [ ]:
annual_q=(1+result_quality.quintile_returns).prod()**(12/len(result_quality.quintile_returns))-1
annual_q.plot(kind='bar',color=['green','lightgray','lightgray','lightgray','red'],title='Quality ? Quintile Returns'); plt.show()

In [ ]:
(1+result_quality.monthly_returns).cumprod().plot(title='Quality Q1 cumulative return'); plt.show()

In [ ]:
pd.Series(result_quality.betas).plot(kind='barh',title='Quality factor decomposition'); plt.show()

In [ ]:
sensitivity=pd.DataFrame(result_quality.sensitivity).T
sensitivity['fragile']=(sensitivity['tstat']-result_quality.alpha_tstat).abs()>0.5
sensitivity

In [ ]:
pd.DataFrame(result_quality.period_results).T

In [ ]:
pd.DataFrame(result_quality.size_terciles or {}).T

In [ ]:
pd.DataFrame(result_quality.sector_breakdown or {}).T

## Quality interpretation
What does this mean? Does it match the hypothesis?

## Accruals
Accruals compare accounting earnings with operating cash flow. We expect low-accrual firms to outperform, with possible stronger effects among smaller firms.

In [ ]:
from strategies.accruals import run as run_accruals
result_accruals = run_accruals(DB_PATH, START, END)

In [ ]:
print('ACCRUALS ? FULL UNIVERSE')
print('Period:', result_accruals.n_months, 'months')
print('Dimensions:', result_accruals.dimensions_used)
print('Alpha:', result_accruals.alpha_annual, 't/p:', result_accruals.alpha_tstat, result_accruals.alpha_pvalue)
print('Betas:', result_accruals.betas)
print('Lookahead violations:', result_accruals.lookahead_violations)

In [ ]:
annual_ac=(1+result_accruals.quintile_returns).prod()**(12/len(result_accruals.quintile_returns))-1
annual_ac.plot(kind='bar',color=['green','lightgray','lightgray','lightgray','red'],title='Accruals ? Quintile Returns'); plt.show()

In [ ]:
pd.concat({'Accruals Q1':(1+result_accruals.monthly_returns).cumprod(),'Equal-weight universe':result_accruals.portfolio_history.groupby('date')['return'].mean().add(1).cumprod()},axis=1).plot(title='Accruals cumulative returns'); plt.show()

In [ ]:
pd.Series(result_accruals.betas).plot(kind='barh',title='Accruals factor decomposition'); plt.show()

In [ ]:
pd.DataFrame(result_accruals.period_results).T

In [ ]:
pd.DataFrame(result_accruals.size_terciles or {}).T

## Three-Signal Summary
This comparison puts the three exploratory signals side by side. It is descriptive only; factor replication and size patterns require interpretation, not automatic promotion.

In [ ]:
def _size_effect(result):
    return 'YES' if result.size_terciles else 'N/A'
comparison = pd.DataFrame([
    {'Signal':'Net share issuance','Alpha':result.alpha_annual,'t-stat':result.alpha_tstat,'RMW load':result.betas.get('RMW',float('nan')),'Size effect':_size_effect(result)},
    {'Signal':'Accruals','Alpha':result_accruals.alpha_annual,'t-stat':result_accruals.alpha_tstat,'RMW load':result_accruals.betas.get('RMW',float('nan')),'Size effect':_size_effect(result_accruals)},
    {'Signal':'Quality','Alpha':result_quality.alpha_annual,'t-stat':result_quality.alpha_tstat,'RMW load':result_quality.betas.get('RMW',float('nan')),'Size effect':_size_effect(result_quality)},
]).set_index('Signal')
comparison

Which signals passed t-stat > 2? Which appear primarily explained by factors? Which show the expected size pattern? Record the honest next step for each.